# Quantum-Guided Cluster Algorithm for Max-Cut

> 🚀 **Skip the local bottleneck.** Qoro is giving away **$100 in free cloud compute credits.**
> Get your API key at **[dash.qoroquantum.net](https://dash.qoroquantum.net)** to run this at scale.

## Why Cloud?

The paper's key result uses **28-node, 10-regular graphs** at QAOA depth **p=5** — that's 28 qubits with deep variational circuits. Even the p=2 transition point is infeasible on a local simulator for problem sizes beyond ~18 qubits. QoroService handles 28+ qubit QAOA with **Maestro's GPU-accelerated MPS simulation**.

**Reference:** [arXiv:2508.10656](https://arxiv.org/abs/2508.10656) — Eder et al., Amazon Quantum Solutions Lab

## Step 0 — Install & Authenticate

```bash
pip install qoro-divi networkx matplotlib numpy
```

In [1]:
import time

import numpy as np

# Set your API key (get one free at https://dash.qoroquantum.net)
# create a .env file
# paste QORO_API_KEY="your_api_key_here"

from algorithm import (
    generate_random_maxcut_graph,
    ising_energy,
    extract_qaoa_correlations,
    coupling_constant_correlations,
    correlation_guided_cluster_algorithm,
    simulated_annealing,
)
from plotting import (
    plot_approximation_ratios,
    plot_correlation_heatmaps,
    plot_circuit_efficiency,
    plot_energy_distribution,
)

from divi.backends import ParallelSimulator

## The Algorithm

Classical methods like simulated annealing make small, local moves that easily get trapped in rugged energy landscapes. The QGCA solves this by using **quantum-derived correlations** to guide cluster formation.

1. **Quantum Phase:** QAOA extracts pairwise correlations ⟨Z_i Z_j⟩ — how spins tend to align in good solutions
2. **Classical Phase:** Cluster Monte Carlo uses these correlations to build meaningful spin clusters, enabling large, coordinated moves through the search space

### Key Divi Features

| Feature | Role |
|---------|------|
| **QAOA** with `GraphProblem.MAXCUT` | Extracts two-point correlations |
| **QWC Observable Grouping** | Up to **60% circuit reduction** |
| **QDrift Trotterization** | Randomized Trotter for scalable depth |
| **QoroService** | Cloud backend for 28+ qubit QAOA |

---

## Phase 1 — Local Toy Problem (10 nodes)

Small graph to prove the algorithm works. Runs in seconds on any laptop.

In [ ]:
from main import run_benchmark

print("Phase 1 — Local Toy Problem (10-node graph)")
print("=" * 50)

t0 = time.time()
results_local = run_benchmark(
    n_nodes=10,
    degree=6,
    qaoa_depths=[1, 2],
    n_iterations_factor=200,
    n_repetitions=10,
    lambda_scale=4,
    seed=42,
    use_cloud=False,
    shots=5_000,
    output_dir="plots",
)
phase1_time = time.time() - t0
print(f"\n⏱️ Phase 1 completed in {phase1_time:.1f}s")

---

## Phase 2 — Paper Benchmark with QoroService (28 nodes)

The paper's primary benchmark: **28-node, 10-regular graphs** at depths p=1,2,3,5. Each QAOA run is dispatched to QoroService.

**Requirements:**
Create a `.env` file in the repo root:
```
QORO_API_KEY="your_api_key_here"
```
👉 **[Get your free API key →](https://dash.qoroquantum.net)**

In [ ]:
print("☁️  Routing 28-qubit QAOA circuits to Qoro Maestro...")

t0 = time.time()
results_cloud = run_benchmark(
    n_nodes=28,
    degree=10,
    qaoa_depths=[1, 2, 3, 5],
    n_iterations_factor=500,
    n_repetitions=30,
    lambda_scale=4,
    seed=42,
    use_cloud=True,
    shots=10_000,
    output_dir="plots",
)
phase2_time = time.time() - t0

print(f"\n⚡ Local  (Phase 1): {phase1_time:.1f}s for 10 nodes")
print(f"⚡ Cloud  (Phase 2): {phase2_time:.1f}s for 28 nodes")

---

## Ready for 100-Node Graphs?

28-qubit QAOA at depth p=5 — impossible locally, trivial on Maestro.

👉 **[Get your free API key at dash.qoroquantum.net](https://dash.qoroquantum.net)** — $100 in credits, no credit card required.